# Post-process a batch of model runs into three xarray datasets

Takes every `.mat` under one S3 tag (a batch submitted by `/tools/coiled/submit.py` from the model extension repo) and writes three Zarr stores:

| store | dims | time | what it holds |
|---|---|---|---|
| `full.zarr` | `cluster × feedstock × depth × time` | annual | every run as it came out of the model, both mineralogies kept apart, plus the E1 and E2 recomputes |
| `point.zarr` | `point × time` | annual | the two mineralogies mixed at the per-state weight `w`, depth collapsed, broadcast onto the deployed points |
| `state.zarr` | `cluster × feedstock × depth × time` | **100/yr** | the raw carbonate state, unreduced — enough to re-derive E1 and E2 without the `.mat` |

- `full.zarr` and `point.zarr` are annual to align with the published deposit.
- `state.zarr` captures the model time resolution because the annual reduction loses information by averaging nonlinear vars. 
- `cluster` is a dimension of `point.zarr` in the sense that every point is tied to some cluster.

In [ ]:
import os, re, warnings
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import numpy as np, pandas as pd, scipy.io as sio, s3fs, xarray as xr
from beerling_analysis import io, mixing, experiments as X

warnings.filterwarnings("ignore")

# Each of these can be set from the environment, so a batch runs without editing
# the notebook -- see the command at the end of the caveats.
TAG        = os.environ.get("BEERLING_TAG", "e0-conus-2gt")   # names the results/ prefix on S3
SCENARIO   = int(os.environ.get("BEERLING_SCENARIO", 2))      # 1 -> 1 Gt/yr, 2 -> 2 Gt/yr; sets which (bas, yy) column of the pickle is the deployment
ROOT       = "s3://carbonplan-carbon-removal/ew-workflows-data/beerling-2025-model-extension"
OUT        = os.environ.get("BEERLING_POSTPROC_OUT", f"{ROOT}/postproc/{TAG}")   # a local path writes locally instead
CACHE      = Path(io.CACHE) / TAG              # .mat staging; ~8.5 GB for the full CONUS batch
KEEP_CACHE = False                             # False -> delete each .mat once read
WORKERS    = 8
WRITE_STATE = os.environ.get("BEERLING_WRITE_STATE", "1") != "0"   # False -> skip state.zarr; saves ~4 GB of RAM (see section 3)

# The CONUS tags name their scenario;
if re.fullmatch(r"e0-conus-\dgt", TAG):
    assert TAG[-3] == str(SCENARIO), f"{TAG} is scenario S{TAG[-3]}, not S{SCENARIO}"

G2T = io.G_PER_M2_TO_T_PER_HA   # g/m2 -> t/ha
fs  = s3fs.S3FileSystem()
CACHE.mkdir(parents=True, exist_ok=True)
print(f"{TAG}  scenario S{SCENARIO}  ->  {OUT}   (state.zarr {'written' if WRITE_STATE else 'NOT written'})")

e0-conus-2gt  scenario S2  ->  s3://carbonplan-carbon-removal/ew-workflows-data/beerling-2025-model-extension/postproc/e0-conus-2gt   (state.zarr NOT written)


## 1. Find the runs

Filenames follow the format `{experiment}_drv{drv}_{state}_{mineralogy}_{tf}yr.mat`. Each file has every field of its job, so the S3 batch is self-describing. `start_year = 2071 − tf` (every deployment ends in 2070). 

In [2]:
PAT = re.compile(r"^(?P<exp>E\d)_drv(?P<drv>\d+)_(?P<source>.+)_(?P<feedstock>[12])_(?P<tf>\d+)yr\.mat$")

runs = pd.DataFrame([
    {**PAT.match(n).groupdict(), "file": n}
    for n in (Path(k).name for k in fs.ls(f"{ROOT}/results/{TAG}"))
    if PAT.match(n)
]).astype({"drv": int, "feedstock": int, "tf": int})
runs["start_year"] = 2071 - runs.tf     # the deployment window always ends 2070
runs = runs.sort_values(["drv", "feedstock"]).reset_index(drop=True)

# dry run: BEERLING_ONLY_CLUSTERS="162,180" keeps only those clusters
ONLY = os.environ.get("BEERLING_ONLY_CLUSTERS")
if ONLY:
    runs = runs[runs.drv.isin([int(x) for x in ONLY.split(",")])].reset_index(drop=True)
    print(f"DRY RUN: clusters {ONLY} only")

pairs = runs.groupby(["drv", "source", "start_year"])["feedstock"].agg(set)
incomplete = pairs[pairs != {1, 2}]

# E1, E2 and the closed background (_cb) need an E0-only batch; otherwise diagnostics only
WITH_E1 = set(runs.exp) == {"E0"}
WITH_E2 = WITH_E1
WITH_CB = WITH_E1

print(f"{len(runs)} runs, {len(pairs)} clusters, experiments {sorted(set(runs.exp))}")
print(f"start years {runs.start_year.min()}-{runs.start_year.max()}, tf {runs.tf.min()}-{runs.tf.max()} yr")
print(f"E1/E2 recompute: {'on' if WITH_E1 else 'OFF -- not an E0-only batch; diagnostics only'}")
print(f"closed background (_cb): {'on' if WITH_CB else 'OFF'}")
if len(incomplete):
    print(f"\n{len(incomplete)} cluster(s) missing a mineralogy -- they cannot be mixed and\n"
          f"will be NaN in point.zarr:\n{incomplete.to_string()}")
runs.head()

676 runs, 338 clusters, experiments ['E0']
start years 2020-2068, tf 3-51 yr
E1/E2 recompute: on
closed background (_cb): on


,exp,drv,source,feedstock,tf,file,start_year
0,E0,1,Michigan,1,25,E0_drv1_Michigan_1_25yr.mat,2046
1,E0,1,Michigan,2,25,E0_drv1_Michigan_2_25yr.mat,2046
2,E0,2,Pennsylvania,1,8,E0_drv2_Pennsylvania_1_8yr.mat,2063
3,E0,2,Pennsylvania,2,8,E0_drv2_Pennsylvania_2_8yr.mat,2063
4,E0,3,Idaho,1,22,E0_drv3_Idaho_1_22yr.mat,2049


## 2. Read one run

A `.mat` holds three grids: the depth-resolved state (100 steps/yr), the mass series (monthly) and `pH` (monthly). Each is reduced to calendar years, dropping year 0:

| kind | reduction |
|---|---|
| cumulative (`cdr`, `*_export`, `P_mass`, `K_mass`) | end of year |
| instantaneous (`pH`, `Ca_*`, `alk`, `q`, `co2_req`) | annual mean |
| log-scaled (`omega*`, `*_ratio`) | annual geometric mean |

- `pH` is the annual mean of the monthly field, which reproduces the published pH.
- `P_mass` and `K_mass` are extra checks against the published deposit in kg/m².
- `p80` and `pH0` (7-layer initial soil pH) are echoed from `inps`. `pH0` matches the published pickle.

### Carbonate block

The model sets pCO2, alkalinity and pH independently, but the carbonate system has only two degrees of freedom. Each projection drops one:

- **`_thermo`, `omega`**: E0's state read through carbonate equilibrium. 
- **`_e1`**: keep pCO2 and alkalinity, solve for pH with a thermodynamic calcite ceiling. 
- **`_e2`**: keep pCO2 and pH, cap exported alkalinity at `carbonate_alk(pH, pCO₂)`.
- **`co2_req`**: keep pH and alkalinity, solve for pCO2.
- **`_cb`**: `pH_thermo` and `co2_req` with the untreated porewater carbonate system closed first (`alk_ctrl_cb = carbonate_alk(pH_ctrl, pCO₂)` replaces the buffer-curve `alk_ctrl`). 

`_thermo`, `omega` and `co2_req` are computed for every batch; `_e1`, `_e2` and `_cb` for E0 only.

- **E1** export is applied as a ratio, `cdr_export_e1 = cdr_export · f_export`, so the published E0 baseline is untouched. Note that `cdr_e1` can exceed `cdr` if enough calcite precipitates.
- **E2** leaves calcite alone (Ω ≈ 1 at the model's buffered pH), so `cdr_e2 = cdr_export_e2 + cdr_precip`. `f_export_e2` is accumulated on the 100/yr grid because `alk_cap` is exponential in pH. 
- **`co2_req`** is an arithmetic mean, like `soil_co2`, and is not clipped (negative means no pCO2 closes the system). `co2_req_ratio` is formed on the 100/yr grid, so it is **not** `co2_req / soil_co2`.

In [ ]:
DEPTH_VARS = ["pH", "Ca_sol", "Ca_sat", "Ca_precip", "alk", "soil_co2"]
COL_VARS   = ["pH_col", "q", "cdr", "cdr_export", "cdr_precip", "c2b", "rock",
              "Ca_export", "Mg_export", "K_export", "Na_export", "P_mass", "K_mass"]

# carbonate block: DIAG_* for every batch; E1_*, E2_*, CB_* (closed background) for E0 only
DIAG_DEPTH = ["Ca_sat_thermo", "pH_thermo", "pH_diag", "omega", "co2_req", "co2_req_ratio"]
E1_DEPTH   = ["Ca_sol_e1", "pH_e1"]
E1_COL     = ["cdr_e1", "cdr_export_e1", "cdr_precip_e1", "Ca_export_e1"]
E2_DEPTH   = ["alk_cap", "alk_added", "pH_ctrl", "omega_buffered"]
E2_COL     = ["cdr_e2", "cdr_export_e2", "f_export_e2", "alk_export_e2"]
CB_DEPTH   = ["alk_ctrl_cb", "pH_cb", "co2_req_cb", "co2_req_cb_ratio"]
# geometric mean over time and depth; co2_req stays arithmetic (it can be negative)
LOG_VARS   = ["omega", "omega_buffered", "co2_req_ratio", "co2_req_cb_ratio"]

ALL_DEPTH = (DEPTH_VARS + DIAG_DEPTH + (E1_DEPTH if WITH_E1 else [])
             + (E2_DEPTH if WITH_E2 else []) + (CB_DEPTH if WITH_CB else []))
ALL_COL   = COL_VARS + (E1_COL if WITH_E1 else []) + (E2_COL if WITH_E2 else [])

# state.zarr: only the E1/E2 inputs. Ca_sat and pH rebuild from these to ~1e-4
# (carb.ca_sat, carb.buffered_pH). Mg/K/Na are concentrations, for E2's alk_ctrl.
STATE_DEPTH = ["Ca_sol", "Ca_precip", "alk", "soil_co2", "Mg_sol", "K_sol", "Na_sol"]
STATE_COL   = ["q", "Ca_export", "Mg_export", "K_export", "Na_export"]

UNITS = dict(pH="", Ca_sol="mol/L", Ca_sat="mol/L", Ca_precip="mol/L", alk="mol/L",
             soil_co2="ppm", pH_col="", q="m/yr", cdr="t CO2/ha", cdr_export="t CO2/ha",
             cdr_precip="t CO2/ha", c2b="t CO2/t rock", rock="t/ha", Ca_export="mol/m2",
             Mg_export="mol/m2", K_export="mol/m2", Na_export="mol/m2",
             P_mass="kg/m2", K_mass="kg/m2",
             Mg_sol="mol/L", K_sol="mol/L", Na_sol="mol/L",
             Ca_sat_thermo="mol/L", pH_thermo="", pH_diag="", omega="",
             co2_req="ppm", co2_req_ratio="x soil_co2",
             Ca_sol_e1="mol/L", pH_e1="", cdr_e1="t CO2/ha", cdr_export_e1="t CO2/ha",
             cdr_precip_e1="t CO2/ha", Ca_export_e1="mol/m2", cdr_total="t CO2",
             cdr_total_e1="t CO2",
             alk_cap="eq/L", alk_added="eq/L", pH_ctrl="", omega_buffered="",
             cdr_e2="t CO2/ha", cdr_export_e2="t CO2/ha", f_export_e2="",
             alk_export_e2="eq/m2", cdr_total_e2="t CO2",
             alk_ctrl_cb="eq/L", pH_cb="", co2_req_cb="ppm", co2_req_cb_ratio="x soil_co2")

def _mean(x, per_yr, tf):
    """Mean over each simulated year, dropping any leading t=0 column."""
    a = np.asarray(x, float)[..., -per_yr * tf:]
    return a.reshape(*a.shape[:-1], tf, per_yr).mean(-1)

def _end(x, per_yr, tf):
    """Value at the end of each simulated year."""
    return np.asarray(x, float)[..., per_yr:per_yr * tf + 1:per_yr]

def _gmean(x, per_yr, tf):
    """Geometric mean over each simulated year, skipping values <= 0 (no Ca yet)."""
    a = np.asarray(x, float)[..., -per_yr * tf:]
    a = a.reshape(*a.shape[:-1], tf, per_yr)
    with np.errstate(divide="ignore", invalid="ignore"):
        return np.exp(np.nanmean(np.where(a > 0, np.log(a), np.nan), -1))

def read_run(name):
    """Fetch one .mat; return (z, start_year, inputs, annual, state).

    `annual` is on calendar years; `state` is on the 100/yr grid, or None.
    """
    p = CACHE / name
    if not p.exists():
        fs.get(f"{ROOT}/results/{TAG}/{name}", str(p))
    try:
        m = sio.loadmat(p, squeeze_me=True, struct_as_record=False)
    finally:
        if not KEEP_CACHE and p.exists():
            p.unlink()

    d, o = m["totals"].diag, m["totals"].Outflow
    tf = int(round(float(np.atleast_1d(d.t)[-1])))
    alk  = np.asarray(d.alk, float)
    rows = [str(x) for x in np.atleast_1d(d.alk_rows)]
    M = lambda x: _mean(x, 100, tf)    # diag grid, instantaneous
    E = lambda x: _end(x, 100, tf)     # diag grid, cumulative
    C = lambda x: _end(x, 12, tf)      # monthly grid, cumulative
    G = lambda x: _gmean(x, 100, tf)   # diag grid, log-scaled

    v = dict(pH=_mean(m["pH"], 12, tf),   # monthly: the published reduction
             Ca_sol=M(d.Ca_sol), Ca_sat=M(d.Ca_sat), Ca_precip=M(d.Ca_precip),
             alk=M(alk[rows.index("alk")]), soil_co2=M(alk[rows.index("CO2")]), q=M(d.q),
             Ca_export=E(o.Ca), Mg_export=E(o.Mg), K_export=E(o.K), Na_export=E(o.Na),
             cdr=C(m["CO2_mass"]) * G2T, cdr_export=C(m["CO2_mass_outflow"]) * G2T,
             cdr_precip=C(m["CO2_mass_precip"]) * G2T, c2b=C(m["c2b_ratio"]),
             # start-of-year value, as c2b_ratio uses (_end is one application ahead)
             rock=np.asarray(m["Application_Basalt_mass"], float)[:12 * tf:12] * G2T,
             P_mass=C(m["P_mass"]), K_mass=C(m["K_mass"]))
    v["pH_col"] = v["pH"].mean(0)     # unweighted over all 31 nodes -- run_simple4.m:39

    # -- carbonate block
    run = X.Run(m)
    try:
        F = X.carbonate_fields(run, with_e1=WITH_E1, with_e2=WITH_E2, with_cb=WITH_CB)
    except AssertionError as e:
        raise AssertionError(f"{name}: {e}") from e
    v.update(Ca_sat_thermo=M(F["Ca_sat_thermo"]), pH_thermo=M(F["pH_thermo"]),
             pH_diag=M(F["pH_diag"]), omega=G(F["omega"]),
             co2_req=M(F["co2_req"]), co2_req_ratio=G(F["co2_req_ratio"]))
    if WITH_CB:
        v.update(alk_ctrl_cb=M(F["alk_ctrl_cb"]), pH_cb=M(F["pH_cb"]),
                 co2_req_cb=M(F["co2_req_cb"]), co2_req_cb_ratio=G(F["co2_req_cb_ratio"]))
    if WITH_E2:
        # for plotting only: their ratio is NOT f_export_e2
        v.update(alk_cap=M(F["alk_cap"]), alk_added=M(F["alk_added"]),
                 pH_ctrl=M(F["pH_ctrl"]), omega_buffered=G(F["omega_buffered"]))
        assert F["frac_cap_negative"] < 0.01, \
            f"{name}: E2 cap is negative at {F['frac_cap_negative']:.1%} of live steps"
        v["f_export_e2"]   = E(F["f_export_e2"])
        v["alk_export_e2"] = E(F["alk_export_e2"])
        v["cdr_export_e2"] = v["cdr_export"] * v["f_export_e2"]
        v["cdr_e2"]        = v["cdr_export_e2"] + v["cdr_precip"]   # calcite unmoved
        gap = np.nanmax(np.abs(v["cdr_e2"] / v["cdr"] - E(F["e2_over_e0"])))
        assert gap < 5e-3, f"{name}: ratio and absolute E2 disagree by {gap:.1%}"
    if WITH_E1:
        assert np.allclose(E(F["sic_e0_t_ha"]), v["cdr_precip"]), \
            f"{name}: our E0 precipitate is not the model's CO2_mass_precip"
        v["cdr_export_e1"] = v["cdr_export"] * E(F["f_export"])
        v["cdr_precip_e1"] = E(F["cdr_precip_e1"])
        v["cdr_e1"]        = v["cdr_export_e1"] + v["cdr_precip_e1"]
        v["Ca_export_e1"]  = E(F["Ca_export_e1"])
        v["Ca_sol_e1"]     = M(F["Ca_sol_e1"])
        v["pH_e1"]         = M(F["pH_e1"])
        # ratio route vs e1()'s absolutes; worst gap over the site runs is 0.2%
        gap = np.nanmax(np.abs(v["cdr_e1"] / v["cdr"] - E(F["e1_over_e0"])))
        assert gap < 5e-3, f"{name}: ratio and absolute E1 disagree by {gap:.1%}"

    assert all(a.shape[-1] == tf for a in v.values()), "the grids disagree on run length"
    assert np.allclose(v["cdr"], v["cdr_export"] + v["cdr_precip"]), "CDR is not export + precip"
    assert np.allclose(v["cdr"] / v["rock"], v["c2b"]), "cdr/rock != the model's own c2b_ratio"

    # -- unreduced state, for state.zarr (t=0 dropped, as in the annual reduction)
    state = None
    if WRITE_STATE:
        state = {k: np.asarray(a, float)[..., -100 * tf:] for k, a in
                 dict(Ca_sol=d.Ca_sol, Ca_precip=d.Ca_precip,
                      alk=alk[rows.index("alk")], soil_co2=alk[rows.index("CO2")],
                      Mg_sol=run.Mg_sol, K_sol=run.K_sol, Na_sol=run.Na_sol,
                      q=d.q, Ca_export=o.Ca, Mg_export=o.Mg,
                      K_export=o.K, Na_export=o.Na).items()}
        assert all(a.shape[-1] == 100 * tf for a in state.values()), "state grid is not 100/yr"
        recon = (state["alk"] - 2 * state["Ca_sol"] - 2 * state["Mg_sol"]
                 - state["K_sol"] - state["Na_sol"])
        assert np.allclose(recon, np.asarray(run.alk_ctrl, float)[..., -100 * tf:]), \
            f"{name}: Mg/K/Na do not reconstruct the alkalinity budget"

    i = m["inps"]
    inputs = dict(p80=float(np.atleast_1d(i.p80).ravel()[0]),
                  crop=str(np.atleast_1d(i.nam).ravel()[0]),
                  pH0=np.asarray(i.pH, float).ravel(),
                  phi=float(F["phi"]), eta=float(F["eta"]),
                  # flagged, not fatal carbonate ceiling collapse (past dAlk ≈ 67 mM, the buffered Ca_sat bisection hits the top of its bracket.)
                  collapsed=float(F.get("frac_ceiling_collapsed", np.nan)))
    return np.atleast_1d(d.z).astype(float), int(np.atleast_1d(m["start_year"]).ravel()[0]), inputs, v, state

z0, s0, i0, v0, st0 = read_run(runs.file.iloc[0])
print(f"{runs.file.iloc[0]}: start {s0}, {v0['cdr'].shape[-1]} yr, {len(z0)} depth nodes "
      f"({z0[0]:.2f}-{z0[-1]:.2f} m), p80 {i0['p80']:.0f} um, {i0['crop']}, "
      f"pH0 {i0['pH0'][0]:.2f}-{i0['pH0'][-1]:.2f} over {len(i0['pH0'])} layers")
print(f"  phi {i0['phi']:.3f}, eta {i0['eta']:.4f}, median annual omega {np.nanmedian(v0['omega']):.0f}")
print(f"  soil pCO2: modelled {np.nanmedian(v0['soil_co2']):,.0f} ppm, required to close its own "
      f"pH and alk {np.nanmedian(v0['co2_req']):,.0f} ppm "
      f"({np.nanmedian(v0['co2_req'])/1e6:.1%} atm, {np.nanmedian(v0['co2_req_ratio']):.0f}x)")
if WITH_E1:
    print(f"  E1/E0 cumulative CDR at 2070: {v0['cdr_e1'][-1] / v0['cdr'][-1]:.4f} "
          f"({v0['cdr'][-1]:.1f} -> {v0['cdr_e1'][-1]:.1f} t CO2/ha)")
if WITH_CB:
    print(f"  closed background: column pH {np.nanmedian(v0['pH_cb'].mean(0)):.2f} "
          f"(whole budget {np.nanmedian(v0['pH_thermo'].mean(0)):.2f}), required soil pCO2 "
          f"{np.nanmedian(v0['co2_req_cb']):,.0f} ppm (whole budget {np.nanmedian(v0['co2_req']):,.0f})")
if WITH_E2:
    print(f"  E2/E0 cumulative CDR at 2070: {v0['cdr_e2'][-1] / v0['cdr'][-1]:.4f} "
          f"({v0['cdr'][-1]:.1f} -> {v0['cdr_e2'][-1]:.1f} t CO2/ha), "
          f"export ratio {v0['f_export_e2'][-1]:.3f}")

E0_drv1_Michigan_1_25yr.mat: start 2046, 25 yr, 31 depth nodes (0.00-0.15 m), p80 100 um, Temperate Maize, pH0 6.37-7.26 over 7 layers
  phi 0.279, eta 0.8517, median annual omega 440
  soil pCO2: modelled 1,766 ppm, required to close its own pH and alk 68,451 ppm (6.8% atm, 35x)
  E1/E0 cumulative CDR at 2070: 0.9018 (246.5 -> 222.3 t CO2/ha)
  closed background: column pH 8.55 (whole budget 9.02), required soil pCO2 32,863 ppm (whole budget 68,451)
  E2/E0 cumulative CDR at 2070: 0.4181 (246.5 -> 103.1 t CO2/ha), export ratio 0.245


## 3. `full.zarr` and `state.zarr` — encompassing every run for each mineralogy

Each run fills its calendar-year slot in a NaN-filled array (which is necessary since runs can start in different years at different sites).

`state.zarr` is filled in the same pass on the 100/yr grid. Its time axis is the fractional calendar year and includes every integer year of `full.zarr`, so `state.sel(time=slice(2069, 2070))` is one year.

Note that `state.zarr` needs ~4 GB of RAM for the CONUS batch (~3 GB on disk).

In [ ]:
clusters = np.array(sorted(runs.drv.unique()))
YEARS    = np.arange(runs.start_year.min(), 2071)
ci       = {d: i for i, d in enumerate(clusters)}
shape    = {v: (len(clusters), 2, len(z0), len(YEARS)) if v in ALL_DEPTH
               else (len(clusters), 2, len(YEARS)) for v in ALL_DEPTH + ALL_COL}
arr      = {v: np.full(s, np.nan) for v, s in shape.items()}
p80_a    = np.full((len(clusters), 2), np.nan)
pH0_a    = np.full((len(clusters), 2, len(i0["pH0"])), np.nan)
phi_a    = np.full((len(clusters), 2), np.nan)
eta_a    = np.full((len(clusters), 2), np.nan)
coll_a   = np.full((len(clusters), 2), np.nan)
crop_a   = np.full((len(clusters), 2), "", dtype=object)

# native axis: fractional calendar year, 0.01 apart; contains YEARS exactly
TSUB = np.round(YEARS[0] - 1 + np.arange(1, 100 * len(YEARS) + 1) / 100, 2)
assert np.isin(YEARS, TSUB).all(), "the annual axis is not a subset of the native one"
sarr = ({v: np.full((len(clusters), 2, len(z0), len(TSUB)) if v in STATE_DEPTH
                    else (len(clusters), 2, len(TSUB)), np.nan)
         for v in STATE_DEPTH + STATE_COL} if WRITE_STATE else {})

def fill(r):
    z, start, inp, v, st = read_run(r.file)
    assert start == r.start_year, f"{r.file}: saved start {start} != derived {r.start_year}"
    assert np.allclose(z, z0), f"{r.file}: different depth grid"
    k = start - YEARS[0]
    for name, a in v.items():
        arr[name][ci[r.drv], r.feedstock - 1, ..., k:k + r.tf] = a
    if st is not None:
        for name, a in st.items():
            sarr[name][ci[r.drv], r.feedstock - 1, ..., 100 * k:100 * (k + r.tf)] = a
    p80_a[ci[r.drv], r.feedstock - 1] = inp["p80"]
    pH0_a[ci[r.drv], r.feedstock - 1] = inp["pH0"]
    phi_a[ci[r.drv], r.feedstock - 1] = inp["phi"]
    eta_a[ci[r.drv], r.feedstock - 1] = inp["eta"]
    coll_a[ci[r.drv], r.feedstock - 1] = inp["collapsed"]
    crop_a[ci[r.drv], r.feedstock - 1] = inp["crop"]

with ThreadPoolExecutor(WORKERS) as ex:
    for i, _ in enumerate(ex.map(fill, runs.itertuples()), 1):
        print(f"  {i}/{len(runs)}", end="\r")

# paired mineralogies share one driver, so must agree on its inputs
both = np.isfinite(p80_a).all(1)
assert np.allclose(p80_a[both, 0], p80_a[both, 1]), "paired runs disagree on p80"
assert np.allclose(pH0_a[both, 0], pH0_a[both, 1]), "paired runs disagree on the initial soil pH"
assert (crop_a[both, 0] == crop_a[both, 1]).all(), "paired runs disagree on the crop"
assert np.allclose(phi_a[both, 0], phi_a[both, 1]), "paired runs disagree on porosity"
assert np.allclose(eta_a[both, 0], eta_a[both, 1]), "paired runs disagree on eta"
p80_c, pH0_c = np.nanmean(p80_a, 1), np.nanmean(pH0_a, 1)
phi_c, eta_c = np.nanmean(phi_a, 1), np.nanmean(eta_a, 1)
crop_c = np.where(crop_a[:, 0] != "", crop_a[:, 0], crop_a[:, 1]).astype(str)

meta = runs.drop_duplicates("drv").set_index("drv").reindex(clusters)
CLUSTER_COORDS = dict(
    source_state=("cluster", meta.source.values),
    start_year=("cluster", meta.start_year.values.astype("int16")),
    w=("cluster", np.array([mixing.weight(s) for s in meta.source])),
    p80=("cluster", p80_c), sim_crop=("cluster", crop_c),
    phi=("cluster", phi_c), eta=("cluster", eta_c))

ds_full = xr.Dataset(
    {v: ((("cluster", "feedstock", "depth", "time") if v in ALL_DEPTH
          else ("cluster", "feedstock", "time")), a, {"units": UNITS[v]})
     for v, a in arr.items()},
    coords=dict(cluster=clusters, feedstock=[1, 2], depth=z0, time=YEARS,
                soil_level=np.arange(1, len(i0["pH0"]) + 1),
                pH0=(("cluster", "soil_level"), pH0_c),
                frac_ceiling_collapsed=(("cluster", "feedstock"), coll_a),
                **CLUSTER_COORDS),
    attrs=dict(tag=TAG, scenario=f"S{SCENARIO}", experiment=", ".join(sorted(set(runs.exp))),
               source=f"{ROOT}/results/{TAG}", e1=str(WITH_E1), e2=str(WITH_E2), cb=str(WITH_CB),
               note="depth is metres below surface; time is the calendar year a value is the "
                    "END of; feedstock 1 = bytownite cluster, 2 = labradorite; p80/sim_crop/pH0 "
                    "are the driver inputs the run was handed, echoed back from inps. "
                    "_thermo and omega read E0's own state through carbonate equilibrium; "
                    "_e1 keeps pCO2 and alkalinity and solves for pH; _e2 keeps pCO2 and pH "
                    "and caps the exported alkalinity; co2_req keeps pH and alkalinity and "
                    "solves for the soil pCO2 that would close them, in ppm, against the "
                    "run's own soil_co2. omega, omega_buffered and co2_req_ratio are "
                    "GEOMETRIC annual means over positive values; co2_req is ARITHMETIC, "
                    "like soil_co2, and is NOT sign-constrained -- negative means no pCO2 "
                    "closes that node-year at all. WARNING: alk_cap and alk_added are "
                    "annual means for plotting -- their ratio is NOT f_export_e2, which is "
                    "accumulated on the 100/yr grid; likewise co2_req/soil_co2 is not "
                    "co2_req_ratio, which is accumulated there too. cdr_precip_e2 is "
                    "absent: it equals cdr_precip. frac_ceiling_collapsed is the fraction "
                    "of live node-times where the model's own buffered Ca_sat ran to its "
                    "bracket floor and sits BELOW the thermodynamic ceiling. "
                    "_cb closes the untreated porewater first (alk_ctrl_cb = "
                    "carbonate_alk(pH_ctrl, soil_co2) replaces the buffer-curve alk_ctrl) and "
                    "solves pH_cb and co2_req_cb on alk_ctrl_cb + alk_added, with no calcite "
                    "step: they are the partners of pH_thermo and co2_req, not of pH_e1. "
                    "co2_req_cb_ratio is GEOMETRIC, like co2_req_ratio."))

ds_state = xr.Dataset(
    {v: ((("cluster", "feedstock", "depth", "time") if v in STATE_DEPTH
          else ("cluster", "feedstock", "time")), a, {"units": UNITS[v]})
     for v, a in sarr.items()},
    coords=dict(cluster=clusters, feedstock=[1, 2], depth=z0, time=TSUB, **CLUSTER_COORDS),
    attrs={**ds_full.attrs,
           "note": "the model's carbonate state, UNREDUCED, on fd15's 100-steps-per-year "
                   "grid; time is the fractional calendar year at the end of each substep, "
                   "so the integer years are exactly full.zarr's axis. This is the store "
                   "E1 re-derives from: del_alk = alk - 2*Ca_sol, and Ca_sat / the model's "
                   "buffered pH follow from carb.ca_sat(soil_co2, del_alk, buffered=True) "
                   "and carb.buffered_pH(alk, soil_co2) to ~1e-4. Mg_sol/K_sol/Na_sol are "
                   "here for E2, which needs alk_ctrl = alk - (2*Ca_sol + 2*Mg_sol + K_sol "
                   "+ Na_sol). All three are CONCENTRATIONS in mol/L; calc_pH.m stores the "
                   "Mg row as 2*Mg and Run.Mg_sol halves it."}) if WRITE_STATE else None
ds_full

  1/676
  2/676
  3/676
  4/676
  5/676


  7/676
  8/676
  9/676
  10/676
  11/676


  13/676
  14/676
  15/676
  16/676
  17/676
  18/676
  19/676
  20/676
  21/676
  22/676
  23/676
  24/676
  25/676


  27/676
  28/676
  29/676


  31/676
  32/676
  33/676


  35/676


... (113 outputs left out)

  658/676


  660/676


  663/676
  664/676
  665/676
  666/676
  667/676
  668/676
  669/676
  670/676
  671/676
  672/676
  673/676
  674/676


<xarray.Dataset> Size: 194MB
Dimensions:                 (cluster: 338, feedstock: 2, depth: 31, time: 51,
                             soil_level: 7)
Coordinates: (12/14)
  * cluster                 (cluster) int64 3kB 1 2 3 4 5 ... 434 435 436 437
    source_state            (cluster) object 3kB 'Michigan' ... 'Minnesota'
    start_year              (cluster) int16 676B 2046 2063 2049 ... 2053 2050
    w                       (cluster) float64 3kB 0.4306 0.8697 ... 0.8295
    p80                     (cluster) float64 3kB 100.0 100.0 ... 100.0 100.0
    sim_crop                (cluster) <U32 43kB 'Temperate Maize' ... 'Temper...
    ...                      ...
  * feedstock               (feedstock) int64 16B 1 2
    frac_ceiling_collapsed  (cluster, feedstock) float64 5kB 0.001419 ... 0.0
  * depth                   (depth) float64 248B 0.0 0.005 0.01 ... 0.145 0.15
  * time                    (time) int64 408B 2020 2021 2022 ... 2068 2069 2070
  * soil_level              (soil_level) int64 56B 1 2 3 4 5 6 7
    pH0                     (cluster, soil_level) float64 19kB 6.371 ... 7.294
Data variables: (12/43)
    pH                      (cluster, feedstock, depth, time) float64 9MB nan...
    Ca_sol                  (cluster, feedstock, depth, time) float64 9MB nan...
    Ca_sat                  (cluster, feedstock, depth, time) float64 9MB nan...
    Ca_precip               (cluster, feedstock, depth, time) float64 9MB nan...
    alk                     (cluster, feedstock, depth, time) float64 9MB nan...
    soil_co2                (cluster, feedstock, depth, time) float64 9MB nan...
    ...                      ...
    cdr_precip_e1           (cluster, feedstock, time) float64 276kB nan ... ...
    Ca_export_e1            (cluster, feedstock, time) float64 276kB nan ... ...
    cdr_e2                  (cluster, feedstock, time) float64 276kB nan ... ...
    cdr_export_e2           (cluster, feedstock, time) float64 276kB nan ... ...
    f_export_e2             (cluster, feedstock, time) float64 276kB nan ... ...
    alk_export_e2           (cluster, feedstock, time) float64 276kB nan ... ...
Attributes:
    tag:         e0-conus-2gt
    scenario:    S2
    experiment:  E0
    source:      s3://carbonplan-carbon-removal/ew-workflows-data/beerling-20...
    e1:          True
    e2:          True
    cb:          True
    note:        depth is metres below surface; time is the calendar year a v...

## 4. Deployment: connecting model columns to points over space

In [5]:
_, key2drv = io.load_drivers()
db = io.load_published("CO2")

pts = pd.DataFrame([
    dict(point=pid, cluster=key2drv[(str(np.atleast_1d(v["nam"]).ravel()[0]), int(v["lind"]))],
         source=str(np.atleast_1d(v["bas"]).ravel()[SCENARIO - 1]).replace(" ", "_"),
         start_year=int(np.atleast_1d(v["yy"]).ravel()[SCENARIO - 1]),
         area_ha=float(v["ar"]), recipient=str(v["nam_l"]),
         crop=str(np.atleast_1d(v["nam"]).ravel()[0]), lind=int(v["lind"]))
    for pid, v in db.items()
    if str(np.atleast_1d(v["bas"]).ravel()[SCENARIO - 1]) != "NaN"   # NaN = never receives rock
]).sort_values("point").reset_index(drop=True)

KEY = ["cluster", "source", "start_year"]
have = runs.rename(columns={"drv": "cluster"})[KEY].drop_duplicates()
deployed, pts = len(pts), pts.merge(have, on=KEY)
print(f"scenario S{SCENARIO}: {len(pts):,} points, {pts.area_ha.sum()/1e6:.2f} M ha, "
      f"{pts.cluster.nunique()} of {len(clusters)} run clusters")
if deployed > len(pts):
    print(f"  {deployed - len(pts):,} deployed points dropped -- their (cluster, source, start) "
          f"is not in this batch")
missing = sorted(set(clusters) - set(pts.cluster))
if missing:
    print(f"  {len(missing)} run cluster(s) with no S{SCENARIO} points: {missing[:8]}")
assert len(pts), f"no S{SCENARIO} point matches any run in {TAG} -- wrong SCENARIO for this batch?"

scenario S2: 3,904 points, 49.18 M ha, 338 of 338 run clusters


## 5. `point.zarr` — mixed, depth-collapsed, per point

$$X(t) = w\,X_1(t) + (1-w)\,X_2(t)$$

applied to every variable, then collapsed over the 31 depth nodes by an unweighted mean (geometric for `LOG_VARS`), as in `run_simple4.m:39`. This reproduces the published pH to 5×10⁻⁵. Points in a cluster are identical per hectare; only area varies.

Note -- compare `pH_e1` with `pH_diag` since they are both on the 100/yr grid. Linear mixing of nonlinear terms means that `omega`, `_thermo` and `_e1` values are approximate when averaged.

In [6]:
w = ds_full.w
mix = w * ds_full.sel(feedstock=1) + (1 - w) * ds_full.sel(feedstock=2)

# depth mean (pH_col already is one); geometric for LOG_VARS
def _collapse(v):
    a = mix[v]
    if v in LOG_VARS:
        with np.errstate(divide="ignore", invalid="ignore"):
            return np.exp(np.log(a.where(a > 0)).mean("depth"))
    return a.mean("depth")

col = mix.drop_dims("depth").assign({v: _collapse(v) for v in ALL_DEPTH if v != "pH"})

ds_point = (col.sel(cluster=pts.cluster.values)
               .rename(cluster="point")
               .assign_coords(point=pts.point.values,
                              cluster=("point", pts.cluster.values),
                              area_ha=("point", pts.area_ha.values),
                              recipient_state=("point", pts.recipient.values),
                              crop=("point", pts.crop.values),
                              lind=("point", pts.lind.values)))
ds_point["cdr_total"] = ds_point.cdr * ds_point.area_ha
if WITH_E1:
    ds_point["cdr_total_e1"] = ds_point.cdr_e1 * ds_point.area_ha
if WITH_E2:
    ds_point["cdr_total_e2"] = ds_point.cdr_e2 * ds_point.area_ha
for v in ds_point.data_vars:
    ds_point[v].attrs["units"] = UNITS[v]
ds_point.attrs = {**ds_full.attrs,
                  "note": "mineralogies mixed at w; depth collapsed by unweighted mean over "
                          "0-0.15 m (omega geometrically); per-hectare unless stated. cluster "
                          "is a coordinate, not a dimension -- the point->cluster map is "
                          "many-to-one. Compare pH_e1 to pH_diag, not to pH_col: pH_col is the "
                          "monthly reduction, pH_diag the 100/yr one pH_e1 is built on. The E2 "
                          "cap was applied at the BASE NODE (fd15.m:495 exports from there) "
                          "before this depth collapse; alk_cap/alk_added here are column means "
                          "and their ratio is not the export ratio. Same for the soil CO2 pair: "
                          "co2_req is a column arithmetic mean (matching soil_co2) and "
                          "co2_req_ratio a column GEOMETRIC one, so the latter is not the "
                          "quotient of the former two -- co2_req spans ~11x over the 31 nodes "
                          "where soil_co2 spans 1.4x. Compare pH_cb with pH_thermo and "
                          "co2_req_cb with co2_req: the same projections, with the untreated "
                          "porewater closed first."}

cum = ds_point.cdr_total.fillna(0.0)
gt = (cum - cum.shift(time=1, fill_value=0.0)).sum("point") / 1e9
print(f"national gross CDR: peak {float(gt.max()):.3f} Gt/yr in {int(gt.idxmax())}, "
      f"cumulative to 2070 {float(cum.sel(time=2070).sum()) / 1e9:.2f} Gt")
e0 = float(cum.sel(time=2070).sum()) / 1e9
if WITH_E1:
    e1 = float(ds_point.cdr_total_e1.fillna(0.0).sel(time=2070).sum()) / 1e9
    print(f"           under E1: {e1:.2f} Gt  ({e1 / e0:.1%} of E0, "
          f"a loss of {e0 - e1:.2f} Gt to pedogenic calcite)")
if WITH_E2:
    e2 = float(ds_point.cdr_total_e2.fillna(0.0).sel(time=2070).sum()) / 1e9
    xp = float((ds_point.cdr_export_e2 * ds_point.area_ha).fillna(0.0).sel(time=2070).sum()) / 1e9
    print(f"           under E2: {e2:.2f} Gt  ({e2 / e0:.1%} of E0, "
          f"a loss of {e0 - e2:.2f} Gt of alkalinity export the modelled pH cannot carry)")
    print(f"             of which export {xp:.2f} Gt; the rest is E0's pedogenic "
          f"calcite, which E2 leaves untouched")
ds_point

national gross CDR: peak 0.461 Gt/yr in 2070, cumulative to 2070 10.58 Gt
           under E1: 9.40 Gt  (88.8% of E0, a loss of 1.18 Gt to pedogenic calcite)
           under E2: 3.84 Gt  (36.3% of E0, a loss of 6.74 Gt of alkalinity export the modelled pH cannot carry)
             of which export 1.40 Gt; the rest is E0's pedogenic calcite, which E2 leaves untouched


<xarray.Dataset> Size: 73MB
Dimensions:           (time: 51, soil_level: 7, point: 3904)
Coordinates: (12/16)
  * time              (time) int64 408B 2020 2021 2022 2023 ... 2068 2069 2070
  * soil_level        (soil_level) int64 56B 1 2 3 4 5 6 7
    pH0               (point, soil_level) float64 219kB 6.371 6.235 ... 7.294
  * point             (point) int64 31kB 1 2 3 4 5 ... 5479 5480 5481 5482 5483
    source_state      (point) object 31kB 'Michigan' 'Michigan' ... 'Minnesota'
    start_year        (point) int16 8kB 2046 2046 2046 2046 ... 2050 2050 2050
    ...                ...
    eta               (point) float64 31kB 0.8517 0.8517 ... 0.8526 0.8526
    cluster           (point) int64 31kB 1 1 1 1 1 1 ... 437 437 437 437 437 437
    area_ha           (point) float64 31kB 1.247e+04 1.138e+04 ... 2.197e+04
    recipient_state   (point) object 31kB 'Wisconsin' 'Wisconsin' ... 'Missouri'
    crop              (point) object 31kB 'Temperate Maize' ... 'Temperate Soy'
    lind              (point) int64 31kB 223684 225220 222917 ... 212173 208338
Data variables: (12/45)
    pH_col            (point, time) float64 2MB nan nan nan ... 7.431 7.251 7.27
    q                 (point, time) float64 2MB nan nan nan ... 0.8391 0.721
    cdr               (point, time) float64 2MB nan nan nan ... 162.9 171.4
    cdr_export        (point, time) float64 2MB nan nan nan ... 113.0 119.0
    cdr_precip        (point, time) float64 2MB nan nan nan ... 49.89 52.39
    c2b               (point, time) float64 2MB nan nan nan ... 0.2037 0.204
    ...                ...
    pH_cb             (point, time) float64 2MB nan nan nan ... 8.441 8.451
    co2_req_cb        (point, time) float64 2MB nan nan ... 3.424e+04 3.433e+04
    co2_req_cb_ratio  (point, time) float64 2MB nan nan nan ... 16.32 15.99
    cdr_total         (point, time) float64 2MB nan nan ... 3.58e+06 3.765e+06
    cdr_total_e1      (point, time) float64 2MB nan nan ... 3.343e+06 3.516e+06
    cdr_total_e2      (point, time) float64 2MB nan nan ... 1.548e+06 1.631e+06
Attributes:
    tag:         e0-conus-2gt
    scenario:    S2
    experiment:  E0
    source:      s3://carbonplan-carbon-removal/ew-workflows-data/beerling-20...
    e1:          True
    e2:          True
    cb:          True
    note:        mineralogies mixed at w; depth collapsed by unweighted mean ...

### Spot-check the mix against the published series

Cumulative CDR and topsoil pH from one shared `w`. Expect errors ~1e-5.

In [7]:
c  = int(pts.cluster.iloc[0])
s  = str(col.source_state.sel(cluster=c).values)
y0 = int(col.start_year.sel(cluster=c).values)
win = slice(y0, 2070)

for flux, ours, tol in [("CO2", col.cdr.sel(cluster=c, time=win), 1e-4),
                        ("pH",  col.pH_col.sel(cluster=c, time=win), 1e-3)]:
    yr, pub = io.published_series(flux, c, s, y0, index=key2drv)
    pub = pub[(yr >= y0) & (yr <= 2070)]
    pub = np.cumsum(pub) if flux == "CO2" else pub     # published CDR is per-year
    err = np.abs(ours.values - pub) / (np.abs(pub) if flux == "CO2" else 1.0)
    print(f"drv{c} {s} {y0}  {flux:3s}  max {'rel' if flux == 'CO2' else 'abs'} "
          f"error vs published N2: {err.max():.2e}")
    assert err.max() < tol, f"{flux} does not reproduce the published series"

drv1 Michigan 2046  CO2  max rel error vs published N2: 1.06e-05
drv1 Michigan 2046  pH   max abs error vs published N2: 3.14e-04


### Readout of the carbonate block across the batch

In [ ]:
if WITH_E1:
    ret = (col.cdr_e1 / col.cdr).sel(time=2070)
    ph0, ph1 = col.pH_diag.mean("time"), col.pH_e1.mean("time")
    om = col.omega.median("time")
    print(f"retained fraction E1/E0 at 2070, over {int(ret.count())} clusters:")
    for p in (5, 25, 50, 75, 95):
        print(f"  p{p:<3d} {float(ret.quantile(p / 100)):.3f}")
    print(f"  min {float(ret.min()):.3f}  max {float(ret.max()):.3f}")
    print(f"\ncolumn-mean porewater pH: E0 {float(ph0.median()):.2f} -> "
          f"E1 {float(ph1.median()):.2f} (median cluster, {float((ph1 - ph0).median()):+.2f})")
    print(f"  clusters above pH 7.5:  E0 {float((ph0 > 7.5).mean()):.0%}  "
          f"E1 {float((ph1 > 7.5).mean()):.0%}")
    print(f"\nOmega at E0's own state (geometric annual mean, median over time):")
    for p in (5, 50, 95):
        print(f"  p{p:<3d} {float(om.quantile(p / 100)):8.1f}")
else:
    om = col.omega.median("time")
    print(f"diagnostics only ({', '.join(sorted(set(runs.exp)))} batch). "
          f"Omega median {float(om.median()):.3f} -- expect ~1 if the ceiling is thermodynamic.")

if WITH_E2:
    ret2 = (col.cdr_e2 / col.cdr).sel(time=2070)
    f2   = col.f_export_e2.sel(time=2070)
    omb  = col.omega_buffered.median("time")
    print(f"\nretained fraction E2/E0 at 2070, over {int(ret2.count())} clusters:")
    for p in (5, 25, 50, 75, 95):
        print(f"  p{p:<3d} {float(ret2.quantile(p / 100)):.3f}  "
              f"(export ratio alone {float(f2.quantile(p / 100)):.3f})")
    print(f"  min {float(ret2.min()):.3f}  max {float(ret2.max()):.3f}")
    print(f"\nOmega at the model's OWN buffered pH: median {float(omb.median()):.4f} "
          f"-- ~1 is why E2 leaves calcite alone.")
    # where the ceiling collapsed, omega_buffered is meaningless
    bad = (ds_full.frac_ceiling_collapsed > 0.5).any("feedstock")
    print(f"  buffered ceiling collapsed over >50% of the column in {int(bad.sum())} of "
          f"{int(bad.count())} clusters"
          + (f": {[int(x) for x in ds_full.cluster[bad].values]} " if int(bad.sum()) else ""))

# required soil CO2: time mean, median cluster
cm, cr = col.soil_co2.mean("time"), col.co2_req.mean("time")
mult   = col.co2_req_ratio.mean("time")
print(f"\nsoil pCO2 needed to close E0's own pH and alkalinity, median cluster:")
print(f"  modelled {float(cm.median()):9,.0f} ppm")
print(f"  required {float(cr.median()):9,.0f} ppm   ({float(cr.median()) / 1e6:.1%} of an "
      f"atmosphere of pure CO2)")
print(f"  multiple {float(mult.median()):9.1f}x    "
      f"(p5 {float(mult.quantile(0.05)):.1f}, p95 {float(mult.quantile(0.95)):.1f})")
fin = int(np.isfinite(ds_full.co2_req).sum())
neg = int((ds_full.co2_req < 0).sum())
if neg:
    print(f"  {neg / fin:.2%} of node-years need a NEGATIVE pCO2 -- there the model's own "
          f"alkalinity is below the floor its pH implies, so no soil CO2 closes them at all")

if WITH_CB:
    pw, pc = col.pH_thermo.mean("time"), col.pH_cb.mean("time")
    rw, rc = col.co2_req.mean("time"), col.co2_req_cb.mean("time")
    print(f"\nclosed background vs whole budget, median cluster, column mean:")
    print(f"  pH        {float(pc.median()):9.2f}   vs {float(pw.median()):.2f}")
    print(f"  soil pCO2 {float(rc.median()):9,.0f}   vs {float(rw.median()):,.0f} ppm")

retained fraction E1/E0 at 2070, over 338 clusters:
  p5   0.800
  p25  0.872
  p50  0.909
  p75  0.964
  p95  1.000
  min 0.752  max 1.001

column-mean porewater pH: E0 7.26 -> E1 8.87 (median cluster, +1.60)
  clusters above pH 7.5:  E0 22%  E1 100%

Omega at E0's own state (geometric annual mean, median over time):
  p5       69.4
  p50     356.1
  p95     536.5

retained fraction E2/E0 at 2070, over 338 clusters:
  p5   0.140  (export ratio alone 0.071)
  p25  0.306  (export ratio alone 0.124)
  p50  0.401  (export ratio alone 0.170)
  p75  0.551  (export ratio alone 0.300)
  p95  0.863  (export ratio alone 0.758)
  min 0.040  max 1.042

Omega at the model's OWN buffered pH: median 0.1805 -- ~1 is why E2 leaves calcite alone.
  buffered ceiling collapsed over >50% of the column in 1 of 338 clusters: [195] -- see notebook 05

soil pCO2 needed to close E0's own pH and alkalinity, median cluster:
  modelled     1,836 ppm
  required   122,808 ppm   (12.3% of an atmosphere of pure CO2)


## 6. Write

All stores go to `OUT` (default `postproc/<tag>/` on S3), overwriting in place. `state.zarr` is chunked by cluster, so `open_zarr(...).sel(cluster=25)` reads a few MB.

In [9]:
stores = [("full", ds_full), ("point", ds_point)]
if WRITE_STATE:
    stores.append(("state", ds_state))

for name, ds in stores:
    # state.zarr: one chunk per cluster (~2.5 MB), whole time axis
    enc = ({v: {"chunks": tuple(1 if d == "cluster" else ds.sizes[d] for d in ds[v].dims)}
            for v in ds.data_vars} if name == "state" else None)
    ds.to_zarr(f"{OUT}/{name}.zarr", mode="w", consolidated=True, encoding=enc)
    n = sum(v.nbytes for v in ds.data_vars.values()) / 1e6
    print(f"{OUT}/{name}.zarr   {dict(ds.sizes)}  {len(ds.data_vars)} vars, {n:.0f} MB uncompressed")

print("\nread back with:")
print(f'  xr.open_zarr("{OUT}/full.zarr")')
if WRITE_STATE:
    print(f'  xr.open_zarr("{OUT}/state.zarr").sel(cluster=...)   # one cluster at a time')

s3://carbonplan-carbon-removal/ew-workflows-data/beerling-2025-model-extension/postproc/e0-conus-2gt/full.zarr   {'cluster': 338, 'feedstock': 2, 'depth': 31, 'time': 51, 'soil_level': 7}  43 vars, 194 MB uncompressed


s3://carbonplan-carbon-removal/ew-workflows-data/beerling-2025-model-extension/postproc/e0-conus-2gt/point.zarr   {'time': 51, 'soil_level': 7, 'point': 3904}  45 vars, 72 MB uncompressed

read back with:
  xr.open_zarr("s3://carbonplan-carbon-removal/ew-workflows-data/beerling-2025-model-extension/postproc/e0-conus-2gt/full.zarr")


## Caveats

- **NaN is meaningful.**  because runs start in different years.
- **`cdr` is gross** and cumulative from first application: `cdr = cdr_export + cdr_precip`, `c2b = cdr / rock`.
- **`full.zarr` and `point.zarr` are annual**, to match the published deposit. `state.zarr` keeps the 100/yr grid.
- **E1 and E2 are post-processes, not model runs.** E1 holds E0's dissolution fixed and changes only speciation. E2 is a bound.
- **The E2 cap is applied at the base node** (z = 0.15 m, `fd15.m:495`), the most generous depth for high CDR.
- **`co2_req` is a diagnostic.** No carbon removal number derives from it.
- **Not stored, because exactly derivable:** `c2b_e1 = cdr_e1 / rock`, `Ca_precip_e1 = Ca_sol + Ca_precip − Ca_sol_e1`, `cdr_precip_e2 = cdr_precip`.
- **`_cb` is a projection, not a run.** With no calcite step, most Ca is supersaturated at `pH_cb` by design. Compare `pH_cb`/`co2_req_cb` with `pH_thermo`/`co2_req`.

## Running it

Every setting in the first cell can come from the environment. From `notebooks/` (call nbconvert through the venv's Python; `.venv/bin/jupyter` has a stale shebang):

```bash
# dry run: two clusters, local output, S3 untouched
BEERLING_TAG=e0-conus-1gt BEERLING_SCENARIO=1 BEERLING_ONLY_CLUSTERS=3,180 \
BEERLING_POSTPROC_OUT=/tmp/pp-dryrun BEERLING_WRITE_STATE=0 \
../.venv/bin/python -m nbconvert --to notebook --execute --output-dir /tmp/pp-dryrun \
    --ExecutePreprocessor.timeout=-1 postprocess-narrow.ipynb

# full batch: overwrites full.zarr and point.zarr under postproc/<tag>/ on S3
BEERLING_TAG=e0-conus-1gt BEERLING_SCENARIO=1 BEERLING_WRITE_STATE=0 \
../.venv/bin/python -m nbconvert --to notebook --execute --inplace \
    --ExecutePreprocessor.timeout=-1 postprocess-narrow.ipynb
```

`BEERLING_WRITE_STATE=0` keeps the existing `state.zarr` and saves ~4 GB of RAM.

# ---